# ROS Bootcamp — Day 1 (Python Vision)
### Face Recognition (InsightFace) + Object Detection (YOLO)

This notebook is written in **tutorial style** (Markdown explanations + runnable code cells).
It is designed for a fresh machine where you will install dependencies, test the webcam, then run:
- **Face detection + embeddings** using **InsightFace**
- **Face recognition** via cosine similarity (enroll known faces)
- **Object detection** using **YOLO (Ultralytics)**

> Tip: Run this on Linux with a working webcam. If you use a headless server, you'll need to stream frames differently (not covered here).

## 0) Setup
### Create a clean environment (recommended)
Run these commands in a terminal (not in the notebook):

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
```

### Install Python packages
```bash
pip install opencv-python numpy matplotlib tqdm
pip install insightface onnxruntime
pip install ultralytics
```

**Notes**
- `onnxruntime` is CPU by default. If you have CUDA properly installed, you can try `onnxruntime-gpu` instead.
- If you get a missing system lib error for OpenCV, install `libgl1` / `libglib2.0-0` on Ubuntu.

In [1]:

# Basic imports
import os
import cv2
import time
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List
import insightface


## 1) Quick webcam sanity check
If you see a moving video feed and can quit with **q**, your webcam pipeline works.

If it fails:
- Try changing `CAM_ID` from 0 to 1.
- On Linux, confirm permission to access `/dev/video*`.
- Close any other program using the camera (Zoom, Teams, browser tab, etc.).

In [2]:

CAM_ID = 0

cap = cv2.VideoCapture(CAM_ID)
if not cap.isOpened():
    raise RuntimeError(f"Could not open camera id {CAM_ID}. Try CAM_ID=1 or check permissions.")

print("Press 'q' to quit.")
while True:
    ok, frame = cap.read()
    if not ok:
        print("Frame read failed.")
        break
    cv2.imshow("Webcam Check", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Press 'q' to quit.


error: OpenCV(4.13.0) /io/opencv/modules/highgui/src/window.cpp:1301: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvShowImage'


## 2) Face detection + embeddings (InsightFace)
We will use InsightFace's `FaceAnalysis`:
- It detects faces
- For each face it returns a **512-D embedding** (feature vector)

We'll then do recognition by comparing embeddings using **cosine similarity**.

### Model choice
We use the common `buffalo_l` model pack (good default).

In [ ]:

from insightface.app import FaceAnalysis

# Initialize InsightFace
# ctx_id = 0 uses GPU if available (with proper onnxruntime-gpu). Use -1 for CPU.
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider","CPUExecutionProvider"])
app.prepare(ctx_id=-1, det_size=(640, 640))

print("InsightFace loaded.")


### Helper functions
- `cosine_sim(a,b)` measures similarity
- `draw_face_box(...)` for visualization

In [ ]:

def cosine_sim(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

def draw_face_box(frame, face, name: str = "", color=(0,255,0)):
    x1, y1, x2, y2 = face.bbox.astype(int)
    cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
    if name:
        cv2.putText(frame, name, (x1, max(20, y1-10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)


## 3) Enroll known faces
We’ll build a small **face database**.

**How enrollment works**
1. Capture a frame from the webcam
2. Detect faces
3. Choose the **largest** face as the subject
4. Store its embedding under a label (e.g., `mostafa`, `alice`)

You can enroll multiple people.

### Controls
- Press **e**: enroll the largest detected face
- Press **q**: quit

In [ ]:

# A simple in-memory face DB: name -> embedding
face_db: Dict[str, np.ndarray] = {}

def largest_face(faces):
    if not faces:
        return None
    areas = [(f, (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1])) for f in faces]
    areas.sort(key=lambda x: x[1], reverse=True)
    return areas[0][0]

CAM_ID = 0
cap = cv2.VideoCapture(CAM_ID)
if not cap.isOpened():
    raise RuntimeError(f"Could not open camera id {CAM_ID}.")

print("Enrollment mode: Press 'e' to enroll, 'q' to quit.")
print("Tip: Put your face centered and well-lit.")

while True:
    ok, frame = cap.read()
    if not ok:
        break

    faces = app.get(frame)
    for f in faces:
        draw_face_box(frame, f, name="face")

    cv2.putText(frame, f"Enrolled: {list(face_db.keys())}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2, cv2.LINE_AA)
    cv2.putText(frame, "Press e=enroll largest face, q=quit", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2, cv2.LINE_AA)

    cv2.imshow("Enroll Faces", frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('e'):
        lf = largest_face(faces)
        if lf is None:
            print("No face detected. Try again.")
            continue
        name = input("Enter label/name for this face (e.g., mostafa): ").strip()
        if not name:
            print("Empty name, skipping.")
            continue
        face_db[name] = lf.embedding.copy()
        print(f"Enrolled '{name}'. Total enrolled: {len(face_db)}")
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print("Done. face_db keys:", list(face_db.keys()))


## 4) Real-time face recognition
We will compare each detected face embedding against the DB.

### Threshold
Cosine similarity is in `[-1, 1]`. For same-person faces, values are typically **high**.
A common starting threshold is **0.35–0.6** depending on camera quality and model.

We’ll use `THRESH=0.45` as a default.

### Controls
- Press **q** to quit

In [ ]:

THRESH = 0.45

if not face_db:
    print("WARNING: face_db is empty. Re-run enrollment cell first.")

def recognize_face(emb: np.ndarray, db: Dict[str, np.ndarray], thresh: float) -> Tuple[str, float]:
    best_name = "unknown"
    best_score = -1.0
    for name, ref in db.items():
        s = cosine_sim(emb, ref)
        if s > best_score:
            best_score = s
            best_name = name
    if best_score < thresh:
        return "unknown", best_score
    return best_name, best_score

CAM_ID = 0
cap = cv2.VideoCapture(CAM_ID)
if not cap.isOpened():
    raise RuntimeError(f"Could not open camera id {CAM_ID}.")

print("Recognition running. Press 'q' to quit.")
while True:
    ok, frame = cap.read()
    if not ok:
        break

    faces = app.get(frame)
    for f in faces:
        name, score = recognize_face(f.embedding, face_db, THRESH)
        label = f"{name} ({score:.2f})"
        draw_face_box(frame, f, name=label, color=(0,255,0) if name!="unknown" else (0,0,255))

    cv2.putText(frame, f"THRESH={THRESH:.2f}  Enrolled={list(face_db.keys())}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2, cv2.LINE_AA)

    cv2.imshow("Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## 5) Object detection (YOLO via Ultralytics)
We'll use **YOLOv8n** (small and fast) for a first demo.

### What you will see
- Boxes + class labels + confidence

### Controls
- Press **q** to quit

In [ ]:

from ultralytics import YOLO

yolo = YOLO("yolov8n.pt")  # downloads weights on first run
print("YOLO ready.")


In [ ]:

CAM_ID = 0
cap = cv2.VideoCapture(CAM_ID)
if not cap.isOpened():
    raise RuntimeError(f"Could not open camera id {CAM_ID}.")

print("YOLO running. Press 'q' to quit.")
while True:
    ok, frame = cap.read()
    if not ok:
        break

    # Ultralytics expects BGR numpy (OpenCV) is fine.
    results = yolo.predict(frame, conf=0.35, verbose=False)[0]

    annotated = results.plot()  # returns BGR image with annotations
    cv2.imshow("YOLOv8 Detection", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## 6) (Optional) Combine: face recognition + YOLO in one loop
This is a common robotics pattern: a single perception loop produces multiple outputs.

We’ll run YOLO and face recognition together and overlay both results.

In [ ]:

THRESH = 0.45

CAM_ID = 0
cap = cv2.VideoCapture(CAM_ID)
if not cap.isOpened():
    raise RuntimeError(f"Could not open camera id {CAM_ID}.")

print("Combined perception running. Press 'q' to quit.")
while True:
    ok, frame = cap.read()
    if not ok:
        break

    # 1) Face recognition
    faces = app.get(frame)
    for f in faces:
        name, score = recognize_face(f.embedding, face_db, THRESH)
        label = f"{name} ({score:.2f})"
        draw_face_box(frame, f, name=label, color=(0,255,0) if name!="unknown" else (0,0,255))

    # 2) YOLO
    results = yolo.predict(frame, conf=0.35, verbose=False)[0]
    frame = results.plot(img=frame)  # draw YOLO on same frame

    cv2.imshow("Face+YOLO", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## Day 1 Wrap-up
By the end of Day 1, students should understand:
- How a webcam frame becomes a numpy array (OpenCV)
- Face embeddings and similarity-based recognition
- Running a pretrained object detector (YOLO)
- How to integrate multiple perception tasks into one loop

Next: Day 2 connects this to **ROS2 topics/services**.